# Zero-shot classification with CLIP — the long version

**Computer Vision in Archaeology Training School, Brno 2026**
Monday morning, *Introduction to computer vision*.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arubrno/atrium-school-ml-lessons/blob/main/1-monday-intro/clip_zero_shot_v2.ipynb)

This is the **spelled-out version** of
[`clip_zero_shot.ipynb`](clip_zero_shot.ipynb). It uses the same images and
labels, and gives the same results. The difference is the code: it's written
step by step, with more comments and fewer Python shortcuts, so that you can
follow what every line does. If you're comfortable with Python, the original is
shorter.

It runs unchanged in Google Colab, on the school JupyterHub, and in any local
Jupyter or IDE. The first code cell works out which of those you're in.

> **Before you change anything**, make it your own copy:
> - **Colab**: *File → Save a copy in Drive*. A notebook opened from GitHub
>   doesn't keep your edits.
> - **School JupyterHub**: copy this notebook from `_atrium-school-ml-lessons/`
>   into your own folder, `/home/jovyan/<your name>/`, and open the copy.

CLIP was trained on image–caption pairs scraped from the web. It never saw an
archaeological training set, and nobody ever told it what an antoninianus is.

You give it an image and a **list of candidate descriptions in plain English**,
and it tells you which description fits best. No training, no annotation, no labels.

This notebook does three things:

1. runs CLIP on a few archaeological photographs;
2. shows that **rewording a label changes the answer**, so the vocabulary is now a parameter;
3. shows that if you offer it **only wrong options, it still picks one, confidently**.

The third point is the one that matters.

### How to read this notebook

- Run the cells **in order, from top to bottom**: click into a cell and press
  **Shift + Enter**. Later cells use things that earlier cells created.
- Lines starting with `#` are **comments**. Python ignores them; they're there
  for you.
- Each code cell starts with a short note on *what it does* above it.

A few words you'll meet along the way:

| Word | Meaning here |
|---|---|
| **model** | A trained neural network: a very long list of numbers ("parameters" or "weights") plus the recipe for using them. |
| **tensor** | PyTorch's name for a block of numbers. It can be a list, a table, or a stack of tables. Its **shape** says how big it is in each direction, e.g. `(1, 3, 224, 224)`. |
| **embedding** | A list of numbers (512 of them for this model) that describes an image or a piece of text. Similar things get similar lists. |
| **logits** | Raw scores that come out of a model. Bigger means a better match, but they aren't probabilities yet. |
| **softmax** | A formula that turns a list of scores into numbers between 0 and 1 that add up to 1. |

## 1. Setup

Run the next cell and read what it prints. It finds the repository (cloning it
first if you're in Colab), installs only the packages that are missing, and
points the model cache somewhere sensible for wherever you're running.

**You don't need to understand the code in this cell.** It's the same in every
notebook of the school and has nothing to do with computer vision. In short, it:

1. looks for the course folder: first where this notebook is, then in the
   folders above it, then in the shared copy on the school hub;
2. downloads the folder with `git clone` if it can't find it (this happens on Colab);
3. tells Python where that folder is (`sys.path`), so that we can `import` our
   own helper files from it;
4. calls `setup(...)`, which installs any missing packages and chooses where
   the downloaded model weights are stored.

Nothing below this cell knows or cares which platform you picked.

<details>
<summary><b>Colab</b>: what to expect</summary>

The clone takes a few seconds, and PyTorch is already installed there. Colab
wipes its disk when the runtime is recycled, so in a new session the 600 MB of
CLIP weights have to be downloaded again. To avoid that, change the last line
of the next cell to `setup("torch", "transformers", drive=True)` before you run
anything else. It asks permission to mount your Google Drive and keeps the
caches in `MyDrive/atrium-school`.

</details>

<details>
<summary><b>School JupyterHub</b>: nothing to set up</summary>

Everyone's server shares the same home directory, `/home/jovyan`. The course
repository is in `_atrium-school-ml-lessons/` and the datasets are in
`_atrium-data/`. The instructors keep both up to date. Work only in your own
folder, `/home/jovyan/<your name>/`: copy the notebook there first, then open
the copy.

The setup cell finds the shared repository from your folder and installs
nothing, because the packages and model weights are already in place. If a
notebook stops responding, use *Kernel → Restart Kernel* and re-run from the top.

</details>

<details>
<summary><b>Your own machine</b>: the one-time install</summary>

Follow the [setup guide](https://arup-cas.github.io/atrium-school-ml/setup.html):
clone the repository, make a virtual environment, and run `pip install -r
requirements.txt` with the PyTorch CPU index. Then start Jupyter anywhere inside
the clone. The setup cell will find the repository around it and install nothing.

</details>

In [ ]:
# --- ATRIUM bootstrap: identical in every notebook of this school ------------
# Colab starts with none of this repository, so this cell has to be able to
# fetch it before it can import anything of ours. Everything else lives in
# atrium_bootstrap.py at the repository root.
import pathlib, subprocess, sys

REPO = "https://github.com/arubrno/atrium-school-ml-lessons.git"
HUB_REPO = pathlib.Path.home() / "_atrium-school-ml-lessons"   # the shared copy on the school hub

here = pathlib.Path.cwd()
root = next((p for p in [here, *here.parents, HUB_REPO]
             if (p / "atrium_bootstrap.py").exists()), None)
if root is None:
    root = pathlib.Path("atrium-school-ml-lessons").resolve()
    if not (root / "atrium_bootstrap.py").exists():
        print("fetching the course repository ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(root)], check=True)

sys.path.insert(0, str(root))
from atrium_bootstrap import setup

# drive=True on Colab keeps the model weights in your Google Drive between sessions.
setup("torch", "transformers");

### Load the tools

**What the next cell does:** it imports the libraries we need. Nothing
visible happens.

A *library* is a collection of ready-made code that somebody else wrote. The
`import` line makes it available here.

In [ ]:
import io                                # lets downloaded bytes be read as if they were a file

import matplotlib.pyplot as plt          # draws pictures and charts
import requests                          # downloads files from the internet (for image URLs)
import torch                             # PyTorch: the deep-learning library that CLIP runs on
from PIL import Image                    # Pillow: opens and converts image files
from transformers import CLIPModel       # the CLIP neural network itself
from transformers import CLIPProcessor   # prepares images and texts the way CLIP expects them

### Load the model

**What the next cell does:** it loads CLIP into memory and prints a short
report. The first time, this downloads about 600 MB of weights. After that
they're read from the cache on disk.

It uses two objects from the `transformers` library, made by Hugging Face:

- the **model** is the neural network, which turns images and texts into embeddings;
- the **processor** is its "reception desk". It resizes each photograph to the
  224 × 224 pixels CLIP was trained on, and it cuts each text into *tokens*
  (word pieces) and turns them into numbers.

In [ ]:
# --- Which model ---------------------------------------------------------------
# Models on the Hugging Face Hub are named "<organisation>/<model name>".
# This is OpenAI's CLIP in its "base" size, which cuts every image into
# 32 x 32 pixel patches ("patch32"). It is small enough to run on a laptop CPU.
MODEL_ID = "openai/clip-vit-base-patch32"

# Which version of the files. A "revision" pins one exact version on the Hub,
# like a commit in Git. This one holds the same weights as the original, saved
# in the newer "safetensors" format (converted by Hugging Face's own bot:
# pull request #66 on the model page). We need it because the original
# pytorch_model.bin file only loads on PyTorch 2.6 or newer, and the school
# JupyterHub has 2.4.
WEIGHTS = "c237dc49a33fc61debc9276459120b7eac67e7ef"


# --- Step 1: load the neural network -------------------------------------------
model = CLIPModel.from_pretrained(MODEL_ID, revision=WEIGHTS, use_safetensors=True)

# --- Step 2: put it in "evaluation" mode -----------------------------------------
# Some kinds of layers behave differently while a model is being trained.
# We are only *using* the model, so we say so. from_pretrained() already did this
# for us; writing it out just makes the intention obvious.
model.eval()

# --- Step 3: load the matching processor ---------------------------------------
processor = CLIPProcessor.from_pretrained(MODEL_ID)


# --- Step 4: a short report --------------------------------------------------------
# Count the parameters: all the learned numbers inside the network.
# model.parameters() gives us the network's weight tensors one by one;
# numel() is "number of elements", i.e. how many numbers are in one tensor.
n_parameters = 0
for weight_tensor in model.parameters():
    n_parameters = n_parameters + weight_tensor.numel()

n_millions = n_parameters / 1_000_000   # the underscores are only there to make it readable

print("Loaded model:", MODEL_ID)
print("Parameters:  ", round(n_millions), "million")
print("Running on:  ", model.device)    # "cpu", because we never move the model to a GPU

> **Before the session.** Run the setup cell and the two cells above it once,
> on whatever you plan to use on the day. The weights download once and are
> then read from the cache: instantly on the hub or your laptop, and on Colab
> too as long as the runtime stays alive. If you skip this, expect a few
> minutes of silence while 600 MB downloads.

## 2. The images

Every dataset this week is reached the same way: `get_dataset(name)`. Where it
actually lives (in the repository, on the hub, or downloaded and cached) is
written down once, in `datasets.yml` at the repository root, so no notebook
needs to know.

The demo set is seven find photographs from the Archaeological Map of the
Czech Republic (AMČR): three sherds, two coins and two brooches. Each one has
the scale bar that every find photograph carries. Licence and IDs are in
`demo-images/CREDITS.md`.

To try your own photographs, see section 8 at the end.

**What the next cell does:** it finds the demo image folder and makes a
*dictionary* called `IMAGES`, which looks up a file by its short name. After
that, `IMAGES["sherd_01"]` gives the path to `sherd_01.jpg`.

In [ ]:
from atrium_data import get_dataset   # our own helper; the setup cell made it importable

# get_dataset() returns the folder that holds the demo images, wherever it is.
DEMO = get_dataset("demo")
print("Image folder:", DEMO)

# Build the dictionary  short name -> path to the file, e.g.
#     "sherd_01"  ->  .../demo-images/sherd_01.jpg
IMAGES = {}                                   # start with an empty dictionary
for path in sorted(DEMO.glob("*.jpg")):       # every .jpg file in the folder, in alphabetical order
    short_name = path.stem                    # the file name without the folder and without ".jpg"
    IMAGES[short_name] = path                 # add one entry to the dictionary

print(len(IMAGES), "images:", list(IMAGES.keys()))

**What the next cell does:** it defines a small *function*, `load()`. A
function is a named piece of code that you can reuse. Defining it doesn't
run anything yet; we call it in the cells below.

`load()` opens a photograph from a file on disk **or** from a web address, and
always gives back the same kind of thing: a Pillow image in RGB colour.

In [ ]:
def load(source):
    """Open an image and return it as a Pillow image in RGB colour.

    `source` can be a path to a file on this computer (a string or a Path),
    or a web address starting with http:// or https://
    """
    source = str(source)   # a Path becomes a plain string, so we can check how it starts

    is_web_address = source.startswith("http://") or source.startswith("https://")

    if is_web_address:
        # Download the file. timeout=30 means: give up after 30 seconds.
        response = requests.get(source, timeout=30)
        # Stop with a clear error if the server answered e.g. "404 Not Found".
        response.raise_for_status()
        # response.content holds the raw bytes of the file. io.BytesIO wraps
        # them so that Pillow can read them as if they were a file on disk.
        image = Image.open(io.BytesIO(response.content))
    else:
        image = Image.open(source)

    # Image files can be greyscale, have a transparency channel (RGBA), and so on.
    # CLIP expects exactly three colour channels: red, green and blue.
    image = image.convert("RGB")
    return image

**What the next cell does:** it draws all seven photographs in a grid, with
their names above them.

In [ ]:
# A grid of 2 rows x 4 columns = 8 panels. In matplotlib a panel is called an "axes".
# figsize is the size of the whole figure in inches: (width, height).
fig, axes = plt.subplots(2, 4, figsize=(12, 5.4))

# `axes` is a 2 x 4 grid of panels. .flat walks through it row by row,
# so here we turn it into one simple list of 8 panels.
panels = list(axes.flat)

# First hide the frame and the tick marks on every panel. We have 7 images for
# 8 panels, so the last panel simply stays empty.
for panel in panels:
    panel.axis("off")

# Then put one image into each panel.
# zip() pairs two lists up item by item: (1st panel, 1st name), (2nd panel, 2nd name), ...
# It stops when the shorter list runs out, so the 8th panel gets nothing.
for panel, short_name in zip(panels, IMAGES.keys()):
    photo = load(IMAGES[short_name])
    panel.imshow(photo)
    panel.set_title(short_name, fontsize=10)

plt.tight_layout()   # tidy up the spacing between the panels
plt.show()

## 3. The whole method, in one function

CLIP puts images and texts into **the same space**. In practice this means it
turns both into embeddings (lists of 512 numbers) that can be compared with
each other. To classify one photograph against your labels, it does four things:

1. **Embed the photograph**, which gives one list of 512 numbers.
2. **Embed every label**, which gives one list of 512 numbers per label.
3. **Compare** the photograph's embedding with each label's embedding, which
   gives one score (a *logit*) per label. A higher score means a closer match.
4. **Softmax**: turn the scores into numbers between 0 and 1 that add up to 1.

Note what step 4 means: **the probabilities are over the labels you supplied.**
They aren't a confidence that the object is really there. Whatever labels
you pass in, the numbers will always add up to 100 %.

**What the next cell does:** it defines `zero_shot()`, which does all four
steps for one image and a list of labels. It also defines `get_probability()`,
a tiny helper that `zero_shot()` uses for sorting.

In [ ]:
def get_probability(pair):
    """Given a (label, probability) pair, return the probability.

    zero_shot() below uses this to sort the pairs by their probability.
    """
    label, probability = pair   # split the pair into its two parts
    return probability


def zero_shot(image, labels):
    """Ask CLIP which of `labels` best describes `image`.

    Returns a list of (label, probability) pairs, best match first, e.g.
        [("a photo of a coin", 0.91), ("a photo of a brooch", 0.06), ...]
    """
    # Step 1: prepare the inputs.
    # The processor resizes the image and turns every label into token numbers.
    #   return_tensors="pt"  ->  give us PyTorch tensors ("pt" = PyTorch)
    #   padding=True         ->  pad the shorter labels so that all labels have
    #                            the same number of tokens (the model needs that)
    inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)

    # Step 2: run the model.
    # torch.no_grad() says "we are not training", so PyTorch skips the extra
    # bookkeeping that learning needs. That makes it faster and saves memory.
    # `inputs` is a dictionary; the two stars (**) pass its entries to the model
    # as named arguments: model(input_ids=..., attention_mask=..., pixel_values=...)
    with torch.no_grad():
        outputs = model(**inputs)

    # Step 3: take the scores (logits).
    # logits_per_image is a table with one row per image and one column per label.
    # We passed a single image, so it has 1 row: shape (1, number of labels).
    logits = outputs.logits_per_image

    # Step 4: turn the scores into probabilities with softmax.
    # dim=1 means "along each row", i.e. across the labels, so every row adds up to 1.
    probabilities = logits.softmax(dim=1)

    # Take row 0 (our only image) and turn it from a tensor into a plain Python list.
    probabilities = probabilities[0].tolist()

    # Pair every label with its probability: [(label, probability), ...]
    pairs = []
    for label, probability in zip(labels, probabilities):
        pairs.append((label, probability))

    # Sort the pairs by probability, highest first.
    #   key=get_probability  ->  compare pairs by the number, not by the text
    #   reverse=True         ->  biggest first
    pairs = sorted(pairs, key=get_probability, reverse=True)

    return pairs

**What the next cell does:** it defines `show()`, which calls `zero_shot()`
and draws the answer: the photograph on the left and one bar per label on the
right. Most of the code is about making the chart look tidy.

In [ ]:
# Two colours for the charts, written as hexadecimal colour codes.
BLUE = "#2c7be5"   # the bars
GREY = "#6b7280"   # the percentages next to the bars


def show(image, labels, title=None):
    """Run zero_shot() and draw the result: photograph left, bar chart right.

    Also returns the result, the sorted list of (label, probability) pairs,
    in case you want to use the numbers.
    """
    # --- 1. Ask the model ----------------------------------------------------
    result = zero_shot(image, labels)

    # --- 2. A figure with two panels side by side ----------------------------
    # plt.subplots gives back the whole figure and its panels; we name the
    # two panels straight away. width_ratios makes the right-hand panel
    # 1.7 times as wide as the left-hand one.
    fig, (ax_image, ax_bars) = plt.subplots(
        1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1, 1.7]}
    )

    # --- 3. Left panel: the photograph -----------------------------------------
    ax_image.imshow(image)
    ax_image.axis("off")   # no frame or tick marks around a photograph

    # --- 4. Right panel: one horizontal bar per label --------------------------
    # barh() draws its FIRST bar at the BOTTOM. Our result is sorted best first,
    # so we go through it in reverse: that way the best label ends up on top.
    bar_labels = []
    bar_values = []
    for label, probability in reversed(result):
        bar_labels.append(label)
        bar_values.append(probability)

    ax_bars.barh(bar_labels, bar_values, color=BLUE, height=0.62)

    # Write the percentage just to the right of each bar.
    # Bar number 0 is at the bottom, bar number 1 above it, and so on.
    for bar_number in range(len(bar_values)):
        value = bar_values[bar_number]
        percentage = f"{value:.0%}"   # the format ":.0%" writes 0.873 as "87%"
        ax_bars.text(value + 0.015, bar_number, percentage,
                     va="center", color=GREY, fontsize=10)

    # --- 5. Cosmetics ----------------------------------------------------------
    ax_bars.set_xlim(0, 1.18)   # leave room to the right of 100 % for the text
    ax_bars.set_xticks([])      # no numbers along the bottom; the bars are labelled
    for side in ["top", "right", "bottom", "left"]:
        ax_bars.spines[side].set_visible(False)   # remove the box around the chart
    if title is not None:
        ax_bars.set_title(title, loc="left", fontsize=11, fontweight="bold")

    plt.tight_layout()
    plt.show()

    return result

### Optional: a look inside `zero_shot()`

*You can skip this part on a first read and come back to it later.*

`zero_shot()` hides the four steps from section 3 inside a single call,
`model(**inputs)`. The next three cells do the same work on one photograph but
stop to print what is happening along the way.

**What the next cell does:** it runs the processor and prints the *shape* of
every tensor it produced.

In [ ]:
example_image = load(IMAGES["coin_01"])
example_labels = ["a photo of a potsherd", "a photo of a coin", "a photo of a brooch"]

inputs = processor(text=example_labels, images=example_image, return_tensors="pt", padding=True)

# `inputs` is a dictionary of tensors. Print each one's name and shape.
for name, tensor in inputs.items():
    print(name, "has shape", tuple(tensor.shape))

How to read those shapes:

- `pixel_values` has shape `(1, 3, 224, 224)`: **1** image, **3** colour
  channels (red, green, blue), **224 × 224** pixels. This is the photograph,
  resized.
- `input_ids` has shape `(3, …)`: **3** labels, each cut into the same number
  of tokens. The shorter ones were padded to match the longest.
- `attention_mask` has the same shape, and marks which tokens are real (1) and
  which are only padding (0).

**What the next cell does:** it runs the model and looks at the embeddings,
and then at steps 3 and 4 (compare, softmax) done by hand.

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

# Steps 1 and 2: the embeddings the model computed.
image_embedding = outputs.image_embeds   # shape (1, 512): 1 image, 512 numbers
text_embeddings = outputs.text_embeds    # shape (3, 512): 3 labels, 512 numbers each
print("image embedding: ", tuple(image_embedding.shape))
print("text embeddings: ", tuple(text_embeddings.shape))
print("the first five numbers describing the photograph:", image_embedding[0, :5].tolist())
print()

# Step 3: compare the photograph with each label.
# CLIP measures how closely two embeddings "point the same way" (cosine
# similarity): 1 would mean identical, 0 unrelated. The embeddings have already
# been scaled to length 1, so this is just "multiply number by number, then add
# everything up".
# CLIP then multiplies every similarity by a fixed factor it learned in training.
scale = model.logit_scale.exp().item()
print("CLIP's scale factor:", round(scale, 1))

my_logits = []
for label_number in range(len(example_labels)):
    similarity = (image_embedding[0] * text_embeddings[label_number]).sum().item()
    my_logits.append(similarity * scale)
    print(f"  similarity {similarity:.3f}  x scale  =  logit {similarity * scale:6.2f}   {example_labels[label_number]}")

print()
print("logits computed by hand: ", [round(x, 2) for x in my_logits])
print("logits from the model:   ", [round(x, 2) for x in outputs.logits_per_image[0].tolist()])

The similarities themselves are small numbers that sit close together. The
scale factor stretches them out, so that a small difference in similarity
becomes a big difference in the logits.

**What the next cell does:** it applies softmax by hand, which is step 4. The
recipe is: raise *e* (≈ 2.718) to the power of each score, then divide each
result by their total.

In [ ]:
import math   # Python's built-in maths functions; we need math.exp()

# Raise e to the power of each logit.
exponentials = []
for logit in my_logits:
    exponentials.append(math.exp(logit))

# Divide each by the total, so that the results add up to 1.
total = sum(exponentials)
my_probabilities = []
for value in exponentials:
    my_probabilities.append(value / total)

print("softmax by hand:")
for label, probability in zip(example_labels, my_probabilities):
    print(f"  {probability:6.1%}   {label}")

print()
print("zero_shot() says:")
for label, probability in zero_shot(example_image, example_labels):
    print(f"  {probability:6.1%}   {label}")

print()
print("The probabilities always add up to", round(sum(my_probabilities), 6))

Two things to take from this:

- **The sum is always 1.** Softmax only shares 100 % out among the labels you
  gave it. It has no way of saying *"none of these"*. Section 6 shows why that
  matters.
- **Softmax exaggerates.** Because of *e* to the power, a lead of 5 points in
  the logits becomes *e*⁵ ≈ 148 times more weight. A small difference in
  similarity therefore comes out looking like a confident answer.

## 4. A sensible set of labels

Five plausible options, one of them right. Two sherds: first a glossy Roman
*terra sigillata* fragment, then a coarse sherd with finger impressions.

Both photographs also contain a scale bar. Watch how much that option gets.

**What the next cell does:** it asks the same five questions about two photographs.

In [ ]:
# Five candidate descriptions. For each sherd, exactly one of them is right.
labels = [
    "a photo of a potsherd",
    "a photo of a coin",
    "a photo of a stone tool",
    "a photo of a scale bar",
    "a photo of a brooch",
]

# The same analysis on two photographs, one after the other.
for short_name in ["sherd_02", "sherd_01"]:
    photo = load(IMAGES[short_name])
    show(photo, labels, "Sensible options: " + short_name)

# Keep the harder photograph, the coarse sherd, in a variable:
# sections 5 and 6 use it again.
img = load(IMAGES["sherd_01"])

## 5. The vocabulary is a parameter

The coarse sherd again (`sherd_01`), with the same model and three ways of
saying the same thing. Watch the numbers move.

There is no "correct" wording. Whatever you choose becomes part of your method,
and it has to go in the paper. (And is *wheel-thrown* even the right
description of this sherd?)

**What the next cell does:** it shows the same photograph three times, each
time with the labels worded differently.

In [ ]:
# A dictionary again: the key is a short name for the style of wording,
# the value is the list of five labels written in that style.
phrasings = {
    "plain": [
        "potsherd",
        "coin",
        "stone tool",
        "scale bar",
        "brooch",
    ],
    "a photo of": [
        "a photo of a potsherd",
        "a photo of a coin",
        "a photo of a stone tool",
        "a photo of a scale bar",
        "a photo of a brooch",
    ],
    "specialist": [
        "a fragment of wheel-thrown pottery",
        "an ancient bronze coin",
        "a knapped flint tool",
        "a photographic scale bar",
        "an ancient bronze brooch (fibula)",
    ],
}

# .items() gives the dictionary's entries as (key, value) pairs.
for style_name, style_labels in phrasings.items():
    show(img, style_labels, "Phrasing: " + style_name)

## 6. Only wrong options

Now take the same sherd and offer the model nothing that fits.

It won't say *"none of these"*, because it has no way to. It will pick the
least-bad option and report a probability that looks just as convincing as a
correct one.

**What the next cell does:** it offers three labels, all of them wrong, and
prints the model's top answer.

In [ ]:
# Three candidate descriptions, none of which is a potsherd.
wrong_only = [
    "a photo of a coin",
    "a photo of a stone tool",
    "a photo of a bicycle",
]

result = show(img, wrong_only, "Only wrong options")

# `result` is sorted best first, so result[0] is the model's top answer:
# a (label, probability) pair, which we split into two variables.
top_label, top_probability = result[0]

# An f-string (f"...") fills in whatever is inside {curly brackets}.
# {top_probability:.0%} writes the number as a percentage, e.g. 0.873 -> "87%".
print()
print(f"The model's answer: '{top_label}' at {top_probability:.0%} confidence.")
print("There is no potsherd option. There is no potsherd in the answer.")

## 7. What to take from this

Zero-shot is a superb way to **explore** a collection you haven't labelled:
triage a folder, find the frames that contain a scale bar, or get a first
answer to *"which of these 8 000 photographs are worth my afternoon"*.

It's a poor way to **report a number**, because:

- the probabilities are over *your* label list, not over reality;
- rewording a label changes the answer;
- there is no "none of the above", so it never reports that something is absent;
- the training data is the open internet, with all of its biases about what
  archaeological material looks like and which parts of the world it comes from.

By Thursday you'll be measuring exactly these failures with precision, recall
and mAP, instead of eyeballing them.

## 8. Try it on your own image

Point it at a photograph (a path or a URL) and give it your own list of
candidate descriptions.

- **Colab**: the folder icon in the left sidebar has an upload button. A file
  you upload lands in `/content`, so `MY_IMAGE = "my_photograph.jpg"` finds it.
- **School JupyterHub**: upload it into your own folder (the ↑ button above the
  file browser), next to your copy of this notebook.
- **Your own machine**: put the file next to this notebook, or give a full path.
- **Anywhere**: a public image URL works too, with no upload at all.

It starts on `coin_02`, a small fragment of a silver coin. Look at how much
*coin* gets. Then try the other demo images by name (`fibula_02` is the spiral
of a spiral brooch), and your own.

Remember what section 6 showed: whatever you leave off the list can't be the answer.

**What the next cell does:** the same as before, but this time you choose the
photograph and the labels. Change the two settings at the top and run it again.

In [ ]:
# --- Your settings --------------------------------------------------------------
# Which photograph? Any of these works:
#     IMAGES["fibula_02"]                      another demo image, by its short name
#     "my_photograph.jpg"                      a file you uploaded
#     "https://example.org/some/photo.jpg"     a public web address
MY_IMAGE = IMAGES["coin_02"]

# Which descriptions should the model choose between? Add, remove or reword them.
MY_LABELS = [
    "a photo of a potsherd",
    "a photo of a coin",
    "a photo of a stone tool",
    "a photo of a brooch",
]


# --- Run -----------------------------------------------------------------------
photo = load(MY_IMAGE)

# Use the file name as the chart title: the part after the last "/".
file_name = str(MY_IMAGE).split("/")[-1]

# show() returns the list of results. We store it in `result`; otherwise
# Jupyter would also print the whole list underneath the chart.
result = show(photo, MY_LABELS, file_name)